In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

n, z = sp.symbols('n z', complex=True)
alpha = sp.Symbol('alpha', real=True)

x_n = alpha**n
X_z = sp.summation(x_n * z**(-n), (n, 0, sp.oo))
X_z_clean = sp.together(X_z.args[0][0]) if X_z.has(sp.Piecewise) else sp.simplify(X_z)

print("=== SYMBOLIC Z-TRANSFORM ===")
display(X_z_clean)

num, den = X_z_clean.as_numer_denom()

def plot_pole_zero(alpha_val):
    num_sub = num.subs(alpha, alpha_val)
    den_sub = den.subs(alpha, alpha_val)
    zeros = np.roots(np.atleast_1d(sp.Poly(num_sub, z).all_coeffs()))
    poles = np.roots(np.atleast_1d(sp.Poly(den_sub, z).all_coeffs()))
    
    plt.figure(figsize=(6, 6))
    ax = plt.subplot(111)
    
    # Dynamic Region of Convergence (ROC): |z| > |alpha|
    r_alpha = abs(alpha_val)
    theta = np.linspace(0, 2*np.pi, 200)
    
    # Outer bound for plotting area
    r_max = 2.0
    r_fill = np.linspace(r_alpha, r_max, 100)
    R, Theta = np.meshgrid(r_fill, theta)
    X_roc = R * np.cos(Theta)
    Y_roc = R * np.sin(Theta)
    
    # Shade the ROC in light green
    ax.contourf(X_roc, Y_roc, R, levels=[r_alpha, r_max], colors=['lightgreen'], alpha=0.4)
    
    # Shade the excluded inner circle (optional, e.g., light red or white)
    if r_alpha > 0:
        r_inner = np.linspace(0, r_alpha, 50)
        R_in, Theta_in = np.meshgrid(r_inner, theta)
        ax.contourf(R_in * np.cos(Theta_in), R_in * np.sin(Theta_in), R_in, levels=[0, r_alpha], colors=['lightcoral'], alpha=0.2)
    
    # Boundary circle of ROC (radius |alpha|)
    ax.plot(r_alpha * np.cos(theta), r_alpha * np.sin(theta), 'g--', linewidth=1.5, label=f'ROC Boundary ($|z| = {r_alpha:.2f}$)')
    
    # Standard Unit Circle
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')
    
    if len(zeros) > 0: ax.scatter(np.real(zeros), np.imag(zeros), s=100, facecolors='none', edgecolors='b', linewidths=2, marker='o', label='Zeros')
    if len(poles) > 0: ax.scatter(np.real(poles), np.imag(poles), s=100, color='r', marker='x', linewidths=3, label='Poles')
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.7)
    ax.set_title(f'Pole-Zero Map & ROC for $x[n] = \\alpha^n u[n]$ ($\\alpha = {alpha_val:.3f}$)', fontweight='bold')
    ax.set_xlabel('Real Part')
    ax.set_ylabel('Imaginary Part')
    ax.legend(loc='upper right', fontsize=9)
    plt.show()

widgets.interactive(plot_pole_zero, alpha_val=widgets.FloatSlider(value=0.5, min=-1.5, max=1.5, step=0.01, description='Alpha:', style={'description_width': 'initial'}))